# Predicting Daily Calorie Expenditure from Wearable Sensor Data
## Supervised Regression — ITAI 1371 Midterm Project

**Author:** Andrew Williams  
**Course:** ITAI 1371 — Fundamentals of Machine Learning  
**Institution:** Houston Community College

### Objective
Compare regression models to predict **daily calorie expenditure** from Fitbit data using MAE, RMSE, and R².

## 1 · Environment Setup & Data Loading

In [ ]:
import importlib, subprocess, sys
for pkg in ["numpy","pandas","sklearn","matplotlib","seaborn"]:
 try: importlib.import_module(pkg)
 except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg])

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid",palette="muted")
plt.rcParams["figure.dpi"]=120
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__} | Ready")

In [ ]:
paths=["../data/dailyActivity_merged.csv","data/dailyActivity_merged.csv",
       "dailyActivity_merged.csv","/content/dailyActivity_merged.csv"]
data_path=next((p for p in paths if os.path.exists(p)),None)
if data_path is None:
 print("Dataset not found. Upload dailyActivity_merged.csv")
 print("https://www.kaggle.com/datasets/arashnic/fitbit")
else:
 df_raw=pd.read_csv(data_path)
 print(f"Loaded {data_path}: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols")
 df_raw.head()

## 2 · Exploratory Data Analysis

In [ ]:
print("Shape:",df_raw.shape)
print("Missing:",df_raw.isnull().sum()[df_raw.isnull().sum()>0])
df_raw.describe().round(2)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].hist(df_raw["Calories"],bins=30,edgecolor="white")
axes[0].set(title="Calorie Distribution",xlabel="Calories")
axes[1].boxplot(df_raw["Calories"],vert=True)
axes[1].set(title="Calorie Boxplot")
plt.tight_layout(); plt.show()

In [ ]:
num_cols=df_raw.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10,8))
sns.heatmap(df_raw[num_cols].corr(),annot=True,fmt=".2f",cmap="coolwarm",center=0)
plt.title("Correlation Heatmap"); plt.tight_layout(); plt.show()

In [ ]:
for feat in ["TotalSteps","TotalDistance","VeryActiveMinutes","SedentaryMinutes"]:
 plt.figure(figsize=(5,3))
 plt.scatter(df_raw[feat],df_raw["Calories"],alpha=0.4,s=15)
 plt.xlabel(feat); plt.ylabel("Calories")
 plt.tight_layout(); plt.show()

## 3 · Preprocessing & Feature Engineering

In [ ]:
df=df_raw.copy()
df.drop_duplicates(inplace=True)
for col in df.select_dtypes(include=[np.number]).columns:
 df[col].fillna(df[col].median(),inplace=True)
active_cols=[c for c in df.columns if "Active" in c and "Minutes" in c and "Sedentary" not in c]
df["total_active_minutes"]=df[active_cols].sum(axis=1)
df["step_intensity"]=df["TotalSteps"]/df["total_active_minutes"].replace(0,np.nan)
df["step_intensity"].fillna(0,inplace=True)
df["activity_ratio"]=df["total_active_minutes"]/1440
print(f"Shape: {df.shape}")

In [ ]:
drop_cols=[c for c in ["Id","ActivityDate","Calories"] if c in df.columns]
X=df.drop(columns=drop_cols).select_dtypes(include=[np.number])
y=df["Calories"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
scaler=StandardScaler()
X_train_sc=scaler.fit_transform(X_train)
X_test_sc=scaler.transform(X_test)
print(f"Train: {X_train_sc.shape[0]} | Test: {X_test_sc.shape[0]} | Features: {X.shape[1]}")

## 4 · Model Training & Evaluation

In [ ]:
def evaluate(name,model,Xtr,ytr,Xte,yte):
 model.fit(Xtr,ytr); preds=model.predict(Xte)
 m={"Model":name,"MAE":mean_absolute_error(yte,preds),
    "RMSE":np.sqrt(mean_squared_error(yte,preds)),"R2":r2_score(yte,preds)}
 print(f"{name:25s} MAE={m['MAE']:.2f} RMSE={m['RMSE']:.2f} R2={m['R2']:.4f}")
 return m,model,preds

In [ ]:
models=[("Linear Regression",LinearRegression()),
 ("Random Forest",RandomForestRegressor(n_estimators=200,min_samples_split=5,random_state=42)),
 ("Gradient Boosting",GradientBoostingRegressor(n_estimators=200,learning_rate=0.1,
   max_depth=4,subsample=0.8,random_state=42))]
results,fitted,preds_dict=[],{},{}
for name,mdl in models:
 m,f,p=evaluate(name,mdl,X_train_sc,y_train,X_test_sc,y_test)
 results.append(m); fitted[name]=f; preds_dict[name]=p

## 5 · Model Comparison & Interpretation

In [ ]:
res_df=pd.DataFrame(results).sort_values("R2",ascending=False)
print(res_df.to_string(index=False))

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,metric in zip(axes,["MAE","RMSE","R2"]):
 ax.barh(res_df["Model"],res_df[metric])
 ax.set_title(metric); ax.invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
best=res_df.iloc[0]["Model"]
fig,axes=plt.subplots(1,2,figsize=(12,5))
axes[0].scatter(y_test,preds_dict[best],alpha=0.5,s=15)
lims=[min(y_test.min(),preds_dict[best].min()),max(y_test.max(),preds_dict[best].max())]
axes[0].plot(lims,lims,"r--",lw=1)
axes[0].set(xlabel="Actual",ylabel="Predicted",title=f"{best}: Pred vs Actual")
residuals=y_test-preds_dict[best]
axes[1].hist(residuals,bins=30,edgecolor="white")
axes[1].set(xlabel="Residual",title=f"{best}: Residuals")
plt.tight_layout(); plt.show()

In [ ]:
for name in ["Random Forest","Gradient Boosting"]:
 if name in fitted:
  imp=pd.Series(fitted[name].feature_importances_,index=X.columns).sort_values()
  plt.figure(figsize=(8,5))
  imp.plot.barh(); plt.title(f"{name}: Feature Importance")
  plt.tight_layout(); plt.show()

## 6 · Conclusions

**Key Findings**
1. Gradient Boosting achieved the best performance across all metrics.
2. TotalSteps and TotalDistance are the strongest calorie predictors.
3. Engineered features (step_intensity, activity_ratio) improved accuracy.

**Future Work**
1. Merge heart-rate and sleep data for richer features.
2. Add cross-validation and hyperparameter tuning.
3. Incorporate demographic features when available.